# Read in tables

In [54]:
import pandas as pd
import numpy as np

In [55]:
CORPUS = pd.read_csv("CORPUS.csv", sep="|")
VOCAB = pd.read_csv("VOCAB.csv", sep="|", index_col="term_str")

# BOW (3)

In [56]:
# Bag at the song level (top level of OHCO)
BAG = ['song_id']

BOW = CORPUS.groupby(BAG + ['term_str']).term_str\
    .count().to_frame('n')

BOW['tfidf'] = 0  # placeholder, filled in after TFIDF step
BOW = BOW.reset_index()
BOW



,song_id,term_str,n,tfidf
0,0,a,3,0
1,0,about,1,0
2,0,ai,3,0
3,0,aisle,1,0
4,0,almost,1,0
...,...,...,...,...
440681,4114,walking,1,0
440682,4114,way,1,0
440683,4114,whisper,1,0
440684,4114,will,1,0


In [57]:
BOW.to_csv("/Users/natalieseah/Desktop/TextasData/BOW.csv", sep="|", index=False)

# DTM (3)

In [58]:
DTM = BOW.set_index(['song_id', 'term_str'])['n'].unstack(fill_value=0)
DTM

term_str,0,00,000,00000,000s,004,005,006,007,008,...,zub,zuboomboom,zucchinis,zurich,zuse,zuzu,zwrotka,zwycizc,zyska,zz
song_id,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4110,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4111,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4112,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# TFIDF (3)

In [59]:
# Number of documents
N = DTM.shape[0]

# Term frequency (relative within each song)
TF = DTM.apply(lambda x: x / x.sum(), axis=1)

# Inverse document frequency
DF = DTM.astype(bool).sum()
IDF = np.log2(N / DF)

# TFIDF
TFIDF = TF * IDF

TFIDF

term_str,0,00,000,00000,000s,004,005,006,007,008,...,zub,zuboomboom,zucchinis,zurich,zuse,zuzu,zwrotka,zwycizc,zyska,zz
song_id,,,,,,,,,,,,,,,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4110,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
4111,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
4112,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0


## Reduced and Normalized TFIDF_L2 (3)

In [60]:
# Keep only significant words (non-stopwords)
sig_words = VOCAB[VOCAB.stop == 0].index
sig_words = sig_words.dropna()
sig_words = sig_words.intersection(TFIDF.columns)

TFIDF_SIG = TFIDF[sig_words]

# L2 normalize
from sklearn.preprocessing import normalize
TFIDF_L2 = pd.DataFrame(
    normalize(TFIDF_SIG, norm='l2'),
    index=TFIDF_SIG.index,
    columns=TFIDF_SIG.columns
)

TFIDF_L2

term_str,0,00,000,00000,000s,004,005,006,007,008,...,zub,zuboomboom,zucchinis,zurich,zuse,zuzu,zwrotka,zwycizc,zyska,zz
song_id,,,,,,,,,,,,,,,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4110,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
4111,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0
4112,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0


In [61]:
TFIDF_L2.to_csv('/Users/natalieseah/Desktop/TextasData/TFIDF_L2.csv', sep='|')